In [19]:
from alphagenome.models import dna_client
import pandas as pd
import numpy as np
from fmannot.config import load_config
from fmannot.data_utils import load_gtf
from fmannot.query.alphagenome import get_variant_effect_prediction
from fmannot.query.alphagenome import get_genome_track_prediction
from fmannot.query.alphagenome import save_all_effect_predictions
from fmannot.query.alphagenome import save_all_track_predictions

from tqdm.notebook import tqdm
tqdm.pandas()

# load api key

In [20]:
config = load_config()

In [21]:
dna_model = dna_client.create(
    api_key = config['alpha_genome']['apikey']
)

# check metadata

In [23]:
output_metadata = dna_model.output_metadata(
    dna_client.Organism.HOMO_SAPIENS
).concatenate()

In [24]:
output_metadata.head()

,name,strand,Assay title,ontology_curie,biosample_name,biosample_type,biosample_life_stage,data_source,endedness,genetically_modified,nonzero_mean,output_type,gtex_tissue,histone_mark,transcription_factor
0,CL:0000084 ATAC-seq,.,ATAC-seq,CL:0000084,T-cell,primary_cell,adult,encode,paired,False,0.739741,OutputType.ATAC,NaN,NaN,NaN
1,CL:0000100 ATAC-seq,.,ATAC-seq,CL:0000100,motor neuron,in_vitro_differentiated_cells,adult,encode,paired,False,0.273136,OutputType.ATAC,NaN,NaN,NaN
2,CL:0000236 ATAC-seq,.,ATAC-seq,CL:0000236,B cell,primary_cell,adult,encode,paired,False,4.700081,OutputType.ATAC,NaN,NaN,NaN
3,CL:0000623 ATAC-seq,.,ATAC-seq,CL:0000623,natural killer cell,primary_cell,adult,encode,paired,False,0.938715,OutputType.ATAC,NaN,NaN,NaN
4,CL:0000624 ATAC-seq,.,ATAC-seq,CL:0000624,"CD4-positive, alpha-beta T cell",primary_cell,adult,encode,paired,False,4.365206,OutputType.ATAC,NaN,NaN,NaN


In [25]:
# checking ontologies
all_ontologies = output_metadata['ontology_curie'].unique().astype(str)

In [26]:
all_ontologies[:10]

array(['CL:0000084', 'CL:0000100', 'CL:0000236', 'CL:0000623',
       'CL:0000624', 'CL:0000625', 'CL:0000787', 'CL:0000788',
       'CL:0000792', 'CL:0000895'], dtype='<U14')

In [27]:
np.sum(all_ontologies == 'nan')

np.int64(1)

In [28]:
output_metadata.loc[output_metadata['ontology_curie'].astype(str) == 'nan', :]

,name,strand,Assay title,ontology_curie,biosample_name,biosample_type,biosample_life_stage,data_source,endedness,genetically_modified,nonzero_mean,output_type,gtex_tissue,histone_mark,transcription_factor
0,donor,+,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,OutputType.SPLICE_SITES,NaN,NaN,NaN
1,acceptor,+,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,OutputType.SPLICE_SITES,NaN,NaN,NaN
2,donor,-,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,OutputType.SPLICE_SITES,NaN,NaN,NaN
3,acceptor,-,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,OutputType.SPLICE_SITES,NaN,NaN,NaN


In [29]:
(output_metadata
    .groupby('output_type')
    .size()
    .rename('# Human tracks')
)

output_type
OutputType.ATAC                  167
OutputType.CAGE                  546
OutputType.DNASE                 305
OutputType.RNA_SEQ               667
OutputType.CHIP_HISTONE         1116
OutputType.CHIP_TF              1617
OutputType.SPLICE_SITES            4
OutputType.SPLICE_SITE_USAGE     734
OutputType.SPLICE_JUNCTIONS      367
OutputType.CONTACT_MAPS           28
OutputType.PROCAP                 12
Name: # Human tracks, dtype: int64

# load external data

In [30]:
# get snps to predict on
snps = pd.read_csv('../data/kunkle_2019_sumstats.csv')
snps.head()

,variant,chr,position_grch37,position_grch38,closest_gene,ref,alt,major,minor,maf,OR,95_CI,Meta_P,I2_Pf,category
0,rs4844610,1,207802552,207629207,CR1,A,C,C,A,0.187,1.17,1.13–1.21,3.6 __10_24,"0, 8 __10_1",known
1,rs6733839,2,127892810,127135234,BIN1,C,T,C,T,0.407,1.20,1.17–1.23,2.1 __10_44,"15, 2 __10_1",known
2,rs10933431,2,233981912,233117202,INPP5D,G,C,C,G,0.223,0.91,0.88–0.94,3.4 __10_9,"0, 8 __10_1",known
3,rs9271058,6,32575406,32607629,HLA-DRB1,A,T,T,A,0.270,1.10,1.07–1.13,1.4 __10_11,"10, 3 __10_1",known
4,rs75932628,6,41129252,41161514,TREM2,C,T,C,T,0.008,2.08,1.73–2.49,2.7 __10_15,"0, 6 __10_1",known


In [31]:
# get ref genome
gtf, gtf_transcripts, transcript_extractor = load_gtf()

In [32]:
# get biosample label annotations
annot = pd.read_csv('../data/annotated_biosample_type.csv', keep_default_na=False)
annot.tail()

,biosample_name,description,high_level_annotation,low_level_annotation
710,left ventricle myocardium,Muscle tissue of the left heart ventricle,Heart,heart_tissue
711,anterior cingulate cortex,Brain region in the limbic system,Brain,brain_tissue_or_region
712,ectocervix,Outer part of the cervix,Reproductive,reproductive_tissue
713,venous blood,Blood from a vein,Blood,blood_tissue
714,nan,Splice site info missing other labels,Other,splice_sites


# query api

## getting variant effect prediction

In [18]:
predicted_effects = snps.progress_apply(get_variant_effect_prediction, args=(dna_model,), axis=1)

  0%|          | 0/22 [00:00<?, ?it/s]

In [35]:
predicted_effects = pd.concat(predicted_effects.tolist())

In [36]:
predicted_effects['variant_scorer']

0        CenterMaskScorer(requested_output=ATAC, width=...
1        CenterMaskScorer(requested_output=ATAC, width=...
2        CenterMaskScorer(requested_output=ATAC, width=...
3        CenterMaskScorer(requested_output=ATAC, width=...
4        CenterMaskScorer(requested_output=ATAC, width=...
                               ...                        
62498    CenterMaskScorer(requested_output=PROCAP, widt...
62499    CenterMaskScorer(requested_output=PROCAP, widt...
62500    CenterMaskScorer(requested_output=PROCAP, widt...
62501    CenterMaskScorer(requested_output=PROCAP, widt...
62502    CenterMaskScorer(requested_output=PROCAP, widt...
Name: variant_scorer, Length: 922902, dtype: object

In [37]:
save_all_effect_predictions(predicted_effects, snps, annot,
                            store_path = '../out/effect_predictions.feather')

## getting genome track prediction

In [20]:
predicted_tracks = snps.progress_apply(get_genome_track_prediction, args=(dna_model,), axis=1)

  0%|          | 0/5 [00:00<?, ?it/s]

# organise and save

In [21]:
save_all_track_predictions(predicted_tracks, snps,
                           store_path = '../out/track_predictions.zarr')

Opening Zarr v3 store 'out/track_predictions.zarr'...


Querying Alpha Genome API:   0%|          | 0/5 [00:00<?, ?it/s]


--- Zarr saving complete ---
